# nb_12 — Semantic Model (Power BI DAX Measures)

**Purpose:** Document the DAX measures to create manually in **Power BI Desktop** after connecting to the MeridianSupplyCo Fabric Lakehouse.

> **Note:** This notebook is documentation only — no PySpark code is executed. All measures must be created in Power BI Desktop after importing the Delta tables via the Microsoft Fabric / OneLake connector.

---

## Step 1 — Connect Power BI to Fabric Lakehouse

1. Open **Power BI Desktop**
2. **Get Data → Microsoft Fabric → Lakehouse**
3. Select the `meridian` Lakehouse and import all Delta tables
4. Verify the star-schema relationships:
   - `FactSales[order_date_key]` → `DimDate[date_key]`
   - `FactSales[customer_key]` → `DimCustomer[customer_key]`
   - `FactSales[product_key]` → `DimProduct[product_key]`
   - `FactSales[salesrep_key]` → `DimSalesRep[salesrep_key]`
   - `FactInvoice[invoice_date_key]` → `DimDate[date_key]`
   - `FactInvoice[customer_key]` → `DimCustomer[customer_key]`
   - `FactInventorySnapshot[product_key]` → `DimProduct[product_key]`

---

## Step 2 — Create DAX Measures

Create a dedicated **Measures** table and add the following four measures:

### Measure 1 — Days Sales Outstanding (DSO)

```dax
DSO =
DIVIDE(
    SUM(FactInvoice[total_amount]),
    CALCULATE(
        SUM(FactInvoice[total_amount]),
        DATESINPERIOD(
            DimDate[date_key],
            LASTDATE(DimDate[date_key]),
            -365,
            DAY
        )
    )
) * 365
```

**Business meaning:** Average number of days to collect payment after a sale.

---

### Measure 2 — Revenue Year-over-Year %

```dax
Revenue YoY% =
DIVIDE(
    [Total Revenue] - [Prior Year Revenue],
    [Prior Year Revenue]
)
```

**Dependencies:** Requires `[Total Revenue]` and `[Prior Year Revenue]` base measures:

```dax
Total Revenue = SUM(FactSales[extended_amount])

Prior Year Revenue =
CALCULATE(
    [Total Revenue],
    SAMEPERIODLASTYEAR(DimDate[date_key])
)
```

---

### Measure 3 — Gross Margin %

```dax
Gross Margin% =
DIVIDE(
    SUM(FactSales[extended_amount]) - SUM(FactSales[cogs]),
    SUM(FactSales[extended_amount])
)
```

**Business meaning:** Percentage of revenue remaining after cost of goods sold.

---

### Measure 4 — Stockout Count

```dax
Stockout Count =
COUNTROWS(
    FILTER(
        FactInventorySnapshot,
        FactInventorySnapshot[on_hand_qty] <= 0
    )
)
```

**Business meaning:** Number of product+warehouse+date combinations where inventory reached zero or below.

---

## Step 3 — Suggested Report Pages

| Page | Key Visuals |
|---|---|
| Executive Summary | Revenue YoY%, Gross Margin%, DSO card |
| Sales Performance | Revenue by territory, by salesrep, trend by month |
| Inventory Health | Stockout Count by product category, on-hand trend |
| Invoice Aging | DSO by customer segment, overdue invoice list |

In [ ]:
# Verification: confirm all expected Delta tables exist in the lakehouse
expected_tables = [
    "DimDate", "DimCurrency", "DimSupplier", "DimProduct",
    "DimCustomer", "DimTerritory", "DimSalesRep", "DimWarehouse",
    "FactInvoice", "FactInvoiceLine", "FactStockMovement",
    "FactInventorySnapshot", "FactSales",
]

existing = {row.tableName for row in spark.sql("SHOW TABLES").collect()}

for t in expected_tables:
    status = "FOUND" if t in existing else "MISSING"
    print(f"{status:8s}  {t}")